<a href="https://colab.research.google.com/github/MitraShabani/Merge-Order-Bias/blob/main/merge_order_coverage_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai matplotlib numpy pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import re
from google.colab import userdata
from openai import OpenAI
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
client = OpenAI()
MODEL_NAME = "gpt-4o-mini"

def generate(prompt, max_new_tokens=300):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
        temperature=0
    )
    return response.choices[0].message.content

RESULTS_DIR = "/content/drive/MyDrive/merge-order-bias/results"
PLOTS_DIR = "/content/drive/MyDrive/merge-order-bias/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

print("Setup ready.")

In [ ]:
# Using existing results to compare
all_files = os.listdir(RESULTS_DIR)
book_ids = set(f.replace("_forward.json", "").replace("_backward.json", "")
               for f in all_files if f.endswith(".json"))

book_pairs = []
for book_id in book_ids:
    has_forward = f"{book_id}_forward.json" in all_files
    has_backward = f"{book_id}_backward.json" in all_files
    if has_forward and has_backward:
        book_pairs.append(book_id)
    else:
        print(f"WARNING: {book_id} is missing forward or backward results — skipping")

print(f"Found {len(book_pairs)} complete book pairs: {book_pairs}")

In [ ]:
# Fact extraction and coverage check
def extract_facts(chapter_summary, n_facts=4):
    prompt = f"""Extract exactly {n_facts} short, specific, checkable facts from the following chapter summary. Each fact should be one short sentence, checkable as true/false, and specific enough that a paraphrase of it would still count (e.g., a named event, a character action, a specific outcome). Do not include vague or generic facts.

Chapter summary:
{chapter_summary}

Output ONLY the {n_facts} facts, one per line, no numbering, no extra text."""
    result = generate(prompt, max_new_tokens=200)
    facts = [line.strip() for line in result.strip().split("\n") if line.strip()]
    return facts[:n_facts]


def fact_present(fact, final_summary):
    prompt = f"""Does the following summary mention or clearly imply this fact (even if paraphrased differently)? Answer with ONLY "yes" or "no".

Fact: {fact}

Summary:
{final_summary}

Answer (yes/no):"""
    result = generate(prompt, max_new_tokens=5).strip().lower()
    return result.startswith("yes")


def compute_coverage(chapter_summaries, final_summary, n_facts=4):
    """For each chapter, extract facts and check how many survive into final_summary.
    Returns a list of dicts: {chapter, facts, coverage}."""
    results = []
    for i, chapter_summary in enumerate(chapter_summaries):
        chapter_num = i + 1
        facts = extract_facts(chapter_summary, n_facts=n_facts)
        found = [fact_present(f, final_summary) for f in facts]
        coverage = sum(found) / len(facts) if facts else 0
        results.append({
            "chapter": chapter_num,
            "facts": facts,
            "found": found,
            "coverage": coverage
        })
        print(f"  Chapter {chapter_num}: {sum(found)}/{len(facts)} facts found (coverage={coverage:.2f})")
    return results

In [ ]:
# Coverage analysis for each book
N_FACTS_PER_CHAPTER = 4
coverage_summary = []

for book_id in book_pairs:
    with open(os.path.join(RESULTS_DIR, f"{book_id}_forward.json"), "r", encoding="utf-8") as f:
        forward = json.load(f)
    with open(os.path.join(RESULTS_DIR, f"{book_id}_backward.json"), "r", encoding="utf-8") as f:
        backward = json.load(f)

    book_title = forward.get("book_title", book_id)
    chapter_summaries = forward["chapter_summaries"]  # same for both directions
    print(f"\n=== {book_title} ===")

    print(" Forward coverage:")
    forward_coverage = compute_coverage(chapter_summaries, forward["final_summary"], N_FACTS_PER_CHAPTER)

    print(" Backward coverage:")
    backward_coverage = compute_coverage(chapter_summaries, backward["final_summary"], N_FACTS_PER_CHAPTER)

    # Save raw coverage data
    coverage_data = {
        "book_title": book_title,
        "forward_coverage": forward_coverage,
        "backward_coverage": backward_coverage
    }
    out_path = os.path.join(RESULTS_DIR, f"{book_id}_coverage.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(coverage_data, f, indent=2, ensure_ascii=False)
    print(f"  Saved: {out_path}")

    # Plot this book
    chapters = [d["chapter"] for d in forward_coverage]
    f_cov = [d["coverage"] for d in forward_coverage]
    b_cov = [d["coverage"] for d in backward_coverage]

    plt.figure(figsize=(10, 6))
    plt.plot(chapters, f_cov, marker='o', label='Forward merge', color='steelblue')
    plt.plot(chapters, b_cov, marker='s', label='Backward merge', color='firebrick')
    plt.xlabel("Chapter number (book order)")
    plt.ylabel("Fact coverage (fraction of chapter's facts found in final summary)")
    plt.title(f"{book_title}: fact coverage by chapter, forward vs. backward")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(chapters)
    plt.ylim(-0.05, 1.05)
    plot_path = os.path.join(PLOTS_DIR, f"coverage_{book_id}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved plot: {plot_path}")

    coverage_summary.append({
        "book": book_title,
        "forward_mean_coverage": np.mean(f_cov),
        "backward_mean_coverage": np.mean(b_cov)
    })

print("\n=== All books processed ===")

In [ ]:
# Cross-book summary table
coverage_df = pd.DataFrame(coverage_summary)
coverage_df.columns = ["Book", "Forward Mean Coverage", "Backward Mean Coverage"]
coverage_table_path = os.path.join(PLOTS_DIR, "coverage_summary_table.csv")
coverage_df.to_csv(coverage_table_path, index=False)
print(f"Saved: {coverage_table_path}")
coverage_df.round(3)